<a href="https://colab.research.google.com/github/skywalker0803r/CFB/blob/main/%E6%BB%BE%E5%8B%95%E5%BC%8F%E9%A0%90%E6%B8%ACy1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import xgboost as xgb
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
import pickle

# --- Data Loading and Preprocessing (as in your original code) ---
df = pd.read_excel('/content/drive/MyDrive/脫硫劑優化改善/20240916-CFB2脫硫劑優化改善.xlsx')
col = df.columns
df = df.iloc[1:,:]
df.columns = col
df = df.set_index('Unnamed: 0')
df.index.name = 'datetime'

for i in df.columns:
    df[i] = pd.to_numeric(df[i], errors='coerce')

coal_low = df[df["MLUT4_FIQ-2BTCF"] < 20]
sox = df["MLUT4_AT-240"]
constant_sox_indices = sox[sox.shift(1) == sox][(sox.shift(2) == sox) & (sox.shift(-1) == sox)].index
constant_sox = df.loc[constant_sox_indices]
common_index = coal_low.index.union(constant_sox.index)
select_df = df.loc[~df.index.isin(common_index), :]

with open("features1.pkl", "rb") as f:
    features = pickle.load(f)

y_col = 'DeSOx_1st'
select_df = select_df[features+[y_col]]

# --- Modeling Code (Corrected Part) ---

def train_model(train_X, train_y):
    model = xgb.XGBRegressor(
        n_estimators=900,
        random_state=42,
        n_jobs=-1,
        learning_rate=0.028,
        max_depth=8,
        subsample=0.75,
        colsample_bytree=0.75,
        objective='reg:squarederror',
        tree_method='hist'
    )
    return model.fit(train_X, train_y)

# 1. 時間排序
select_df = select_df.sort_index().reset_index(drop=True)

# 2. 設定
target_col = "DeSOx_1st"
time_windows_len = 100 #<--- 調整這裡

# 加入前一期目標欄位當特徵
select_df["prev_target"] = select_df[target_col].shift(1)

# 刪除因 shift 產生的 NaN 資料
select_df = select_df.dropna().reset_index(drop=True)

# 定義特徵欄位除了時間跟target_col不能用其他都可以用
feature_cols = [col for col in select_df.columns if col not in ['timestamp', target_col]]

# 開始訓練的索引
start_idx = time_windows_len

# 計算total_steps
total_steps = len(select_df) - start_idx

# 3. 儲存預測結果
predictions = []
abs_errors = []
thresholds = []
indices = []

# To store recent errors for threshold calculation for the *current* model
current_model_recent_errors = []
threshold_update_window = 100 # Calculate threshold from last 100 errors
percentile_for_threshold = 90 # 90th percentile of recent errors

# 4. 初始化進度條
pbar = tqdm(total=total_steps)

# 5. 動態預測迴圈
i = start_idx
error_exceeded_threshold = True  # Start with training a model

while i < len(select_df):
    # 早停判斷<不得移除>
    if pbar.n >= 1500: # This means it will stop after 1500 predictions, not 1000 training steps.
        break

    # 判斷是否要建模 並記錄
    if 'current_model' not in locals() or error_exceeded_threshold:
        train_df = select_df.iloc[i - time_windows_len:i]
        train_X = train_df[feature_cols]
        train_y = train_df[target_col]

        # train model start
        current_model = train_model(train_X, train_y)

        # Reset recent errors for the new model
        current_model_recent_errors = []
        # For the first prediction after re-training, we won't have enough recent errors
        # to calculate a meaningful threshold. We can set a default or wait.
        # Here, we'll simply let the first few errors build up before calculating.
        current_threshold = np.inf # Set a very high threshold initially for a newly trained model

        error_exceeded_threshold = False # Reset flag

    # 預測i
    test_row = select_df.iloc[i]
    test_X = test_row[feature_cols].values.reshape(1, -1)
    true_y = test_row[target_col]
    pred_y = current_model.predict(test_X)[0]
    error = abs(pred_y - true_y)

    # Add the current error to the list of recent errors for this model
    current_model_recent_errors.append(error)

    # If we have enough recent errors, calculate the threshold for the *current* model's performance
    if len(current_model_recent_errors) >= threshold_update_window:
        current_threshold = np.percentile(current_model_recent_errors[-threshold_update_window:], percentile_for_threshold)
    elif len(current_model_recent_errors) > 0: # If less than window size, use all available errors
        current_threshold = np.percentile(current_model_recent_errors, percentile_for_threshold)
    else:
        current_threshold = np.inf # Should ideally not happen after first few predictions

    # 紀錄
    predictions.append(pred_y)
    abs_errors.append(error)
    thresholds.append(current_threshold)
    indices.append(i)

    # 更新進度條
    pbar.update(1)

    # 根據預測i的結果判斷是否要設置error_exceeded_threshold
    # The check for re-training now uses the threshold calculated from actual recent test errors
    if error > current_threshold:
        error_exceeded_threshold = True
        # If error exceeds, a new model will be trained in the next iteration
        # and 'i' increments by 1
        i += 1
    else:
        # If error does not exceed, stay with the current model for the next prediction
        # 'i' increments by 1, and the loop will try to predict 'i+1' with the *same* model
        i += 1
        error_exceeded_threshold = False # Continue using current model

# 結束訓練
pbar.close()

# 6. 輸出結果表
result_df = select_df.loc[indices].copy()
result_df['prediction'] = predictions
result_df['abs_error'] = abs_errors
result_df['threshold'] = thresholds

# 預覽結果
print(result_df.head())

from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# 指標評估
y_true = result_df['DeSOx_1st']
y_pred = result_df['prediction']

# Calculate R-squared
r2 = r2_score(y_true, y_pred)
# Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
# Calculate MAPE
mape = np.mean(np.abs((y_true - y_pred) / y_true[y_true != 0])) * 100 if np.any(y_true != 0) else 0


# 假設 r2, rmse, mape 是事先計算好的指標
metrics = {
    'Metric': ['R-squared', 'RMSE', 'MAPE'],
    'Value': [r2, rmse, mape],
    'Unit': ['', '', '%']
}

df_metrics = pd.DataFrame(metrics)
df_metrics

  3%|▎         | 1500/53611 [03:26<1:59:28,  7.27it/s]

     MLUT4_AIA792101B  MLUT4_TE-252D  MLUT4_AT-232A  MLUT4_FIC-231B  \
100          6.900000     857.535397     155.517031             0.0   
101          6.867849     859.417612     157.352135             0.0   
102          6.805979     860.947126     157.096622             0.0   
103          6.800000     859.555342     155.481862             0.0   
104          6.800000     856.837148     154.581236             0.0   

     MLUT4_TE-252F  MLUT4_TE-252I  MLUT4_PIC-233  MLUT4_FIC-231C      前爐SOx濃度  \
100     874.123596     880.261472     971.592529       16.858142  2117.256243   
101     874.862420     881.222624     971.485716       16.856887  2127.408170   
102     874.890518     883.643240     969.908187       16.855632  2129.567383   
103     872.808587     882.925170     970.941871       16.854377  2122.962504   
104     871.799639     882.827232     974.520517       16.853122  2119.318169   

     MLUT4_TE-252G  ...  MLUT4_FQ-205  MLUT4_TE-251F  MLUT4_TE-252E  \
100     843.126

,Metric,Value,Unit
0,R-squared,0.896728,
1,RMSE,0.004349,
2,MAPE,0.319640,%


# 針對篩掉的值建模

In [8]:
import pandas as pd
import xgboost as xgb
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
import pickle

# --- Data Loading and Preprocessing (as in your original code) ---
df = pd.read_excel('/content/drive/MyDrive/脫硫劑優化改善/20240916-CFB2脫硫劑優化改善.xlsx')
col = df.columns
df = df.iloc[1:,:]
df.columns = col
df = df.set_index('Unnamed: 0')
df.index.name = 'datetime'

for i in df.columns:
    df[i] = pd.to_numeric(df[i], errors='coerce')

coal_low = df[df["MLUT4_FIQ-2BTCF"] < 20]
sox = df["MLUT4_AT-240"]
constant_sox_indices = sox[sox.shift(1) == sox][(sox.shift(2) == sox) & (sox.shift(-1) == sox)].index
constant_sox = df.loc[constant_sox_indices]
common_index = coal_low.index.union(constant_sox.index)
select_df = df.loc[df.index.isin(common_index), :]

with open("features1.pkl", "rb") as f:
    features = pickle.load(f)

y_col = 'DeSOx_1st'
select_df = select_df[features+[y_col]]

# --- Modeling Code (Corrected Part) ---

# 將inf -inf都轉成nan
def train_model(train_X, train_y):
    train_X = pd.DataFrame(train_X).replace([np.inf, -np.inf], np.nan)
    train_y = pd.Series(train_y).replace([np.inf, -np.inf], np.nan)
    model = xgb.XGBRegressor(
        n_estimators=900,
        random_state=42,
        n_jobs=-1,
        learning_rate=0.028,
        max_depth=8,
        subsample=0.75,
        colsample_bytree=0.75,
        objective='reg:squarederror',
        tree_method='hist'
    )
    return model.fit(train_X, train_y)

# 1. 時間排序
select_df = select_df.sort_index().reset_index(drop=True)
# 或是打亂
#select_df = select_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 2. 設定
target_col = "DeSOx_1st"
time_windows_len = 100 #<--- 調整這裡

# 加入前一期目標欄位當特徵
select_df["prev_target"] = select_df[target_col].shift(1)

# 刪除因 shift 產生的 NaN 資料
select_df = select_df.dropna().reset_index(drop=True)

# 定義特徵欄位除了時間跟target_col不能用其他都可以用
feature_cols = [col for col in select_df.columns if col not in ['timestamp', target_col]]

# 開始訓練的索引
start_idx = time_windows_len

# 計算total_steps
total_steps = len(select_df) - start_idx

# 3. 儲存預測結果
predictions = []
abs_errors = []
thresholds = []
indices = []

# To store recent errors for threshold calculation for the *current* model
current_model_recent_errors = []
threshold_update_window = 100 # Calculate threshold from last 100 errors
percentile_for_threshold = 90 # 90th percentile of recent errors

# 4. 初始化進度條
pbar = tqdm(total=total_steps)

# 5. 動態預測迴圈
i = start_idx
error_exceeded_threshold = True  # Start with training a model

while i < len(select_df):
    # 早停判斷<不得移除>
    if pbar.n >= 1500: # This means it will stop after 1500 predictions, not 1000 training steps.
        break

    # 判斷是否要建模 並記錄
    if 'current_model' not in locals() or error_exceeded_threshold:
        train_df = select_df.iloc[i - time_windows_len:i]
        train_X = train_df[feature_cols]
        train_y = train_df[target_col]

        # train model start
        current_model = train_model(train_X, train_y)

        # Reset recent errors for the new model
        current_model_recent_errors = []
        # For the first prediction after re-training, we won't have enough recent errors
        # to calculate a meaningful threshold. We can set a default or wait.
        # Here, we'll simply let the first few errors build up before calculating.
        current_threshold = np.inf # Set a very high threshold initially for a newly trained model

        error_exceeded_threshold = False # Reset flag

    # 預測i
    test_row = select_df.iloc[i]
    test_X = test_row[feature_cols].values.reshape(1, -1)
    true_y = test_row[target_col]
    pred_y = current_model.predict(test_X)[0]
    error = abs(pred_y - true_y)

    # Add the current error to the list of recent errors for this model
    current_model_recent_errors.append(error)

    # If we have enough recent errors, calculate the threshold for the *current* model's performance
    if len(current_model_recent_errors) >= threshold_update_window:
        current_threshold = np.percentile(current_model_recent_errors[-threshold_update_window:], percentile_for_threshold)
    elif len(current_model_recent_errors) > 0: # If less than window size, use all available errors
        current_threshold = np.percentile(current_model_recent_errors, percentile_for_threshold)
    else:
        current_threshold = np.inf # Should ideally not happen after first few predictions

    # 紀錄
    predictions.append(pred_y)
    abs_errors.append(error)
    thresholds.append(current_threshold)
    indices.append(i)

    # 更新進度條
    pbar.update(1)

    # 根據預測i的結果判斷是否要設置error_exceeded_threshold
    # The check for re-training now uses the threshold calculated from actual recent test errors
    if error > current_threshold:
        error_exceeded_threshold = True
        # If error exceeds, a new model will be trained in the next iteration
        # and 'i' increments by 1
        i += 1
    else:
        # If error does not exceed, stay with the current model for the next prediction
        # 'i' increments by 1, and the loop will try to predict 'i+1' with the *same* model
        i += 1
        error_exceeded_threshold = False # Continue using current model

# 結束訓練
pbar.close()

# 6. 輸出結果表
result_df = select_df.loc[indices].copy()
result_df['prediction'] = predictions
result_df['abs_error'] = abs_errors
result_df['threshold'] = thresholds

# 預覽結果
print(result_df.head())

from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# 指標評估
y_true = result_df['DeSOx_1st']
y_pred = result_df['prediction']

def evaluate_model(y_true, y_pred):
    # 轉成 numpy 陣列並清除 inf 和 nan
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # 把 inf/-inf 轉成 nan
    y_true = np.where(np.isfinite(y_true), y_true, np.nan)
    y_pred = np.where(np.isfinite(y_pred), y_pred, np.nan)

    # 找出合法的索引（兩者皆為有效數值）
    valid_idx = ~np.isnan(y_true) & ~np.isnan(y_pred)

    # 過濾合法值
    y_true = y_true[valid_idx]
    y_pred = y_pred[valid_idx]

    # 安全檢查
    if len(y_true) == 0:
        raise ValueError("❌ 沒有合法的 y_true/y_pred 資料可供評估")

    # R-squared
    r2 = r2_score(y_true, y_pred)

    # RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # MAPE（避免除以零錯誤）
    non_zero_mask = y_true != 0
    if np.any(non_zero_mask):
        mape = np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100
    else:
        mape = np.nan  # 或設成 0

    # 用 DataFrame 呈現
    df = pd.DataFrame({
        "Metric": ["R-squared", "RMSE", "MAPE"],
        "Value": [round(r2, 4), round(rmse, 4), round(mape, 4)],
        "Unit": ["", "", "%"]
    })

    return df
evaluate_model(y_true, y_pred)

 90%|████████▉ | 1500/1671 [02:12<00:15, 11.28it/s] 

     MLUT4_AIA792101B  MLUT4_TE-252D  MLUT4_AT-232A  MLUT4_FIC-231B  \
100          5.000098     903.514237     184.451813       12.612293   
101          4.800148     905.396745     161.996383       12.611442   
102          4.767596     905.606454     181.059039       12.610592   
103          4.863756     904.526081     180.408984       12.609742   
104          4.959915     904.366163     181.471045       12.608892   

     MLUT4_TE-252F  MLUT4_TE-252I  MLUT4_PIC-233  MLUT4_FIC-231C      前爐SOx濃度  \
100     880.908382     890.024001    1235.132720       12.654865  1970.726303   
101     882.736726     892.693854    1229.383167       12.699685  1970.893201   
102     883.591318     895.155770    1216.227896       12.698413  1971.060098   
103     884.606529     896.496090    1211.624981       12.697142  1971.226995   
104     882.468656     893.789713    1211.098914       12.695870  1971.393890   

     MLUT4_TE-252G  ...  MLUT4_FQ-205  MLUT4_TE-251F  MLUT4_TE-252E  \
100     860.052

,Metric,Value,Unit
0,R-squared,-12.2113,
1,RMSE,179.0382,
2,MAPE,5088.0257,%


In [11]:
import pandas as pd
import xgboost as xgb
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
import pickle

# --- Data Loading and Preprocessing (as in your original code) ---
df = pd.read_excel('/content/drive/MyDrive/脫硫劑優化改善/20240916-CFB2脫硫劑優化改善.xlsx')
col = df.columns
df = df.iloc[1:,:]
df.columns = col
df = df.set_index('Unnamed: 0')
df.index.name = 'datetime'

for i in df.columns:
    df[i] = pd.to_numeric(df[i], errors='coerce')

coal_low = df[df["MLUT4_FIQ-2BTCF"] < 20]
sox = df["MLUT4_AT-240"]
constant_sox_indices = sox[sox.shift(1) == sox][(sox.shift(2) == sox) & (sox.shift(-1) == sox)].index
constant_sox = df.loc[constant_sox_indices]
common_index = coal_low.index.union(constant_sox.index)
select_df = df.loc[df.index.isin(common_index), :]

with open("features1.pkl", "rb") as f:
    features = pickle.load(f)

y_col = 'DeSOx_1st'
select_df = select_df[features+[y_col]]

# --- Modeling Code (Corrected Part) ---

# 將inf -inf都轉成nan
def train_model(train_X, train_y):
    model = xgb.XGBRegressor(
        n_estimators=900,
        random_state=42,
        n_jobs=-1,
        learning_rate=0.028,
        max_depth=8,
        subsample=0.75,
        colsample_bytree=0.75,
        objective='reg:squarederror',
        tree_method='hist'
    )
    return model.fit(train_X, train_y)

# 1. 時間排序
#select_df = select_df.sort_index().reset_index(drop=True)
# 或是打亂
select_df = select_df.sample(frac=1, random_state=42).reset_index(drop=True).replace([np.inf, -np.inf], np.nan).dropna()

# 2. 設定
target_col = "DeSOx_1st"
time_windows_len = 100 #<--- 調整這裡

# 加入前一期目標欄位當特徵
select_df["prev_target"] = select_df[target_col].shift(1)

# 刪除因 shift 產生的 NaN 資料
select_df = select_df.dropna().reset_index(drop=True)

# 定義特徵欄位除了時間跟target_col不能用其他都可以用
feature_cols = [col for col in select_df.columns if col not in ['timestamp', target_col]]

# 開始訓練的索引
start_idx = time_windows_len

# 計算total_steps
total_steps = len(select_df) - start_idx

# 3. 儲存預測結果
predictions = []
abs_errors = []
thresholds = []
indices = []

# To store recent errors for threshold calculation for the *current* model
current_model_recent_errors = []
threshold_update_window = 100 # Calculate threshold from last 100 errors
percentile_for_threshold = 90 # 90th percentile of recent errors

# 4. 初始化進度條
pbar = tqdm(total=total_steps)

# 5. 動態預測迴圈
i = start_idx
error_exceeded_threshold = True  # Start with training a model

while i < len(select_df):
    # 早停判斷<不得移除>
    if pbar.n >= 1500: # This means it will stop after 1500 predictions, not 1000 training steps.
        break

    # 判斷是否要建模 並記錄
    if 'current_model' not in locals() or error_exceeded_threshold:
        train_df = select_df.iloc[i - time_windows_len:i]
        train_X = train_df[feature_cols]
        train_y = train_df[target_col]

        # train model start
        current_model = train_model(train_X, train_y)

        # Reset recent errors for the new model
        current_model_recent_errors = []
        # For the first prediction after re-training, we won't have enough recent errors
        # to calculate a meaningful threshold. We can set a default or wait.
        # Here, we'll simply let the first few errors build up before calculating.
        current_threshold = np.inf # Set a very high threshold initially for a newly trained model

        error_exceeded_threshold = False # Reset flag

    # 預測i
    test_row = select_df.iloc[i]
    test_X = test_row[feature_cols].values.reshape(1, -1)
    true_y = test_row[target_col]
    pred_y = current_model.predict(test_X)[0]
    error = abs(pred_y - true_y)

    # Add the current error to the list of recent errors for this model
    current_model_recent_errors.append(error)

    # If we have enough recent errors, calculate the threshold for the *current* model's performance
    if len(current_model_recent_errors) >= threshold_update_window:
        current_threshold = np.percentile(current_model_recent_errors[-threshold_update_window:], percentile_for_threshold)
    elif len(current_model_recent_errors) > 0: # If less than window size, use all available errors
        current_threshold = np.percentile(current_model_recent_errors, percentile_for_threshold)
    else:
        current_threshold = np.inf # Should ideally not happen after first few predictions

    # 紀錄
    predictions.append(pred_y)
    abs_errors.append(error)
    thresholds.append(current_threshold)
    indices.append(i)

    # 更新進度條
    pbar.update(1)

    # 根據預測i的結果判斷是否要設置error_exceeded_threshold
    # The check for re-training now uses the threshold calculated from actual recent test errors
    if error > current_threshold:
        error_exceeded_threshold = True
        # If error exceeds, a new model will be trained in the next iteration
        # and 'i' increments by 1
        i += 1
    else:
        # If error does not exceed, stay with the current model for the next prediction
        # 'i' increments by 1, and the loop will try to predict 'i+1' with the *same* model
        i += 1
        error_exceeded_threshold = False # Continue using current model

# 結束訓練
pbar.close()

# 6. 輸出結果表
result_df = select_df.loc[indices].copy()
result_df['prediction'] = predictions
result_df['abs_error'] = abs_errors
result_df['threshold'] = thresholds

# 預覽結果
print(result_df.head())

from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# 指標評估
y_true = result_df['DeSOx_1st']
y_pred = result_df['prediction']

def evaluate_model(y_true, y_pred):
    # 轉成 numpy 陣列並清除 inf 和 nan
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # 把 inf/-inf 轉成 nan
    y_true = np.where(np.isfinite(y_true), y_true, np.nan)
    y_pred = np.where(np.isfinite(y_pred), y_pred, np.nan)

    # 找出合法的索引（兩者皆為有效數值）
    valid_idx = ~np.isnan(y_true) & ~np.isnan(y_pred)

    # 過濾合法值
    y_true = y_true[valid_idx]
    y_pred = y_pred[valid_idx]

    # 安全檢查
    if len(y_true) == 0:
        raise ValueError("❌ 沒有合法的 y_true/y_pred 資料可供評估")

    # R-squared
    r2 = r2_score(y_true, y_pred)

    # RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # MAPE（避免除以零錯誤）
    non_zero_mask = y_true != 0
    if np.any(non_zero_mask):
        mape = np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100
    else:
        mape = np.nan  # 或設成 0

    # 用 DataFrame 呈現
    df = pd.DataFrame({
        "Metric": ["R-squared", "RMSE", "MAPE"],
        "Value": [round(r2, 4), round(rmse, 4), round(mape, 4)],
        "Unit": ["", "", "%"]
    })

    return df
evaluate_model(y_true, y_pred)


  0%|          | 0/88 [03:01<?, ?it/s]

 91%|█████████ | 1500/1646 [06:10<00:36,  4.05it/s]

     MLUT4_AIA792101B  MLUT4_TE-252D  MLUT4_AT-232A  MLUT4_FIC-231B  \
100          7.282373     903.510678       0.000000        1.393147   
101          4.300000     964.726470     224.542651       10.358447   
102         11.573350     833.039827     188.082050       15.428135   
103          7.200000     932.874387     203.964266       24.511201   
104          5.038086     901.202799     143.627605       16.961439   

     MLUT4_TE-252F  MLUT4_TE-252I  MLUT4_PIC-233  MLUT4_FIC-231C      前爐SOx濃度  \
100     895.795865     817.946246     608.739983       10.077881  1213.320357   
101     931.647992     920.712180     881.195664       14.626470  2050.484784   
102     773.959713     770.825417     487.417607        0.098234   916.970614   
103     900.356269     929.785504     895.200324        0.045014  1944.363860   
104     908.493986     776.459055    1166.974705       14.626251  1629.168807   

     MLUT4_TE-252G  ...  MLUT4_FQ-205  MLUT4_TE-251F  MLUT4_TE-252E  \
100     840.733

,Metric,Value,Unit
0,R-squared,-0.2713,
1,RMSE,54.9805,
2,MAPE,95.8249,%
